In [2]:
import pandas as pd
import numpy as np
import os

file_path = "DataCoSupplyChainDataset.csv"

df = pd.read_csv(file_path, encoding="cp1252")

print("Dataset Shape:", df.shape)
print(df.columns.tolist())
df.head()

Dataset Shape: (180519, 53)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Statu

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
print(df.info())

print(df[[
    'Order Id',
    'order date (DateOrders)',
    'Category Name',
    'Customer Segment',
    'Shipping Mode',
    'Sales',
    'Order Profit Per Order',
    'Late_delivery_risk'
]].head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [4]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)

df['Weekday'] = df['order date (DateOrders)'].dt.day_name()

weekday_analysis = df.groupby('Weekday').agg(
    Total_Orders=('Order Id', 'nunique'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Order Profit Per Order', 'sum'),
    Late_Delivery_Risk=('Late_delivery_risk', 'mean')
).reset_index()

weekday_analysis['Late_Delivery_Risk'] *= 100

weekday_order = [
    'Monday', 'Tuesday', 'Wednesday', 'Thursday',
    'Friday', 'Saturday', 'Sunday'
]

weekday_analysis['Weekday'] = pd.Categorical(
    weekday_analysis['Weekday'],
    categories=weekday_order,
    ordered=True
)

weekday_analysis = weekday_analysis.sort_values('Weekday')

weekday_analysis

,Weekday,Total_Orders,Total_Sales,Total_Profit,Late_Delivery_Risk
1,Monday,9368,5.280298e+06,612331.430216,55.053130
5,Tuesday,9395,5.195252e+06,544637.390415,54.300991
6,Wednesday,9397,5.088116e+06,544067.980202,54.758276
4,Thursday,9395,5.237198e+06,520938.490699,55.005436
0,Friday,9400,5.306701e+06,564628.251019,55.170685
2,Saturday,9430,5.376950e+06,596359.610799,54.395583
3,Sunday,9367,5.300220e+06,583939.820699,55.114469


In [5]:
category_analysis = df.groupby('Category Name').agg(
    Total_Orders=('Order Id', 'nunique'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Order Profit Per Order', 'sum'),
    Late_Delivery_Risk=('Late_delivery_risk', 'mean')
).reset_index()

category_analysis['Late_Delivery_Risk'] *= 100

category_analysis = category_analysis.sort_values(
    'Total_Sales', ascending=False
)

category_analysis

,Category Name,Total_Orders,Total_Sales,Total_Profit,Late_Delivery_Risk
18,Fishing,15164,6.929654e+06,756220.767190,54.926407
12,Cleats,20386,4.431943e+06,494636.919791,54.971284
9,Camping & Hiking,12299,4.118426e+06,427455.568106,54.534198
10,Cardio Equipment,11355,3.694843e+06,383011.098485,54.496677
47,Women's Apparel,17869,3.147800e+06,350421.029567,54.556691
46,Water Sports,13758,3.113845e+06,325146.960038,54.806950
34,Men's Footwear,18783,2.891758e+06,311902.820214,54.486200
30,Indoor/Outdoor Games,16623,2.888994e+06,318451.430554,54.746606
38,Shop By Sport,10136,1.309522e+06,129813.960315,55.152950
13,Computers,442,6.630000e+05,69656.810171,50.678733


In [7]:
def analyze_segment(data, column):
    result = data.groupby(column, dropna=False).agg(
        Total_Orders=('Order Id', 'nunique'),
        Total_Sales=('Sales', 'sum'),
        Total_Profit=('Order Profit Per Order', 'sum'),
        Late_Delivery_Risk=('Late_delivery_risk', 'mean')
    ).reset_index()

    result['Late_Delivery_Risk'] *= 100

    return result.sort_values('Total_Sales', ascending=False)


customer_analysis = analyze_segment(df, 'Customer Segment')
shipping_analysis = analyze_segment(df, 'Shipping Mode')
market_analysis = analyze_segment(df, 'Market')

print("Customer Segment Analysis")
display(customer_analysis)

print("Shipping Mode Analysis")
display(shipping_analysis)

print("Market Analysis")
display(market_analysis)

Customer Segment Analysis


,Customer Segment,Total_Orders,Total_Sales,Total_Profit,Late_Delivery_Risk
0,Consumer,34119,1.909579e+07,2.073488e+06,54.808350
1,Corporate,19856,1.116841e+07,1.202575e+06,54.722663
2,Home Office,11777,6.520538e+06,6.908403e+05,55.070440


Shipping Mode Analysis


,Shipping Mode,Total_Orders,Total_Sales,Total_Profit,Late_Delivery_Risk
3,Standard Class,39324,2.202239e+07,2.370454e+06,38.071683
2,Second Class,12778,7.145445e+06,7.503082e+05,76.632781
0,First Class,10079,5.674370e+06,6.431219e+05,95.322499
1,Same Day,3571,1.942529e+06,2.030184e+05,45.743042


Market Analysis


,Market,Total_Orders,Total_Sales,Total_Profit,Late_Delivery_Risk
1,Europe,18561,1.087240e+07,1.169443e+06,55.207753
2,LATAM,17181,1.027761e+07,1.123322e+06,54.355158
3,Pacific Asia,17577,8.273744e+06,8.577534e+05,55.046049
4,USCA,8579,5.066529e+06,5.643138e+05,54.800574
0,Africa,3854,2.294453e+06,2.520712e+05,54.589289


In [8]:
print(df['Delivery Status'].value_counts(dropna=False))

Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64


In [9]:
df['Is_Late_Delivery'] = (
    df['Delivery Status'].astype(str).str.strip().str.lower()
    == 'late delivery'
)

issue_analysis = df.groupby('Category Name').agg(
    Total_Orders=('Order Id', 'nunique'),
    Late_Delivery_Records=('Is_Late_Delivery', 'sum'),
    Total_Sales=('Sales', 'sum')
).reset_index()

issue_analysis['Late_Delivery_Rate_%'] = (
    issue_analysis['Late_Delivery_Records']
    / issue_analysis['Total_Orders'].replace(0, np.nan)
) * 100

issue_analysis = issue_analysis.sort_values(
    'Late_Delivery_Rate_%', ascending=False
)

issue_analysis

,Category Name,Total_Orders,Late_Delivery_Records,Total_Sales,Late_Delivery_Rate_%
23,Golf Bags & Carts,61,42,1.036939e+04,68.852459
12,Cleats,20386,13496,4.431943e+06,66.202296
34,Men's Footwear,18783,12121,2.891758e+06,64.531757
47,Women's Apparel,17869,11476,3.147800e+06,64.222956
30,Indoor/Outdoor Games,16623,10565,2.888994e+06,63.556518
18,Fishing,15164,9516,6.929654e+06,62.753891
46,Water Sports,13758,8517,3.113845e+06,61.905800
9,Camping & Hiking,12299,7487,4.118426e+06,60.874868
32,Lacrosse,341,206,3.946479e+04,60.410557
10,Cardio Equipment,11355,6805,3.694843e+06,59.929546
